In [ ]:
import os
import yaml

# --- 1. INSTALLAZIONE LIBRERIA UFFICIALE YOLO ---
# Ultralytics supporta nativamente YOLOv9 (modelli yolov9c o yolov9e)
os.system("pip install ultralytics")

In [ ]:
import os
import yaml
from ultralytics import YOLO

print("=== PREPARAZIONE AMBIENTE DI TEST E ESTRAZIONE METRICHE ===\n")

# --- 1. PERCORSI ---
# !!! ATTENZIONE !!!
# modificare i path
percorso_modello_best = "/inserisci/il/path/per/best.pt"
# Il percorso in input dove si trova la cartella 'images' e 'labels' dello split3
percorso_base_dataset = "/inserisci/il/path/alla/cartella/split3"

# --- 2. RICREIAMO I FILE YAML PER IL TEST ---
yaml_test_normal = {
    "path": percorso_base_dataset, 
    "train": "images/train", 
    "val": "images/test_normal", 
    "nc": 2, 
    "names": ["Volto", "Targa"]
}

yaml_test_fisheye = {
    "path": percorso_base_dataset, 
    "train": "images/train", 
    "val": "images/test_fisheye", 
    "nc": 2, 
    "names": ["Volto", "Targa"]
}

# Salviamo gli yaml nella cartella working temporanea
with open("/inserisci/il/path/di/output/per/dataset_test_normal_ciclo3.yaml", "w") as f: 
    yaml.dump(yaml_test_normal, f, sort_keys = False)
    
with open("/inserisci/il/path/di/output/per/dataset_test_fisheye_ciclo3.yaml", "w") as f: 
    yaml.dump(yaml_test_fisheye, f, sort_keys = False)

# --- 3. CARICAMENTO DEL MODELLO ---
print(f"Caricamento modello da: {percorso_modello_best}...")
modello = YOLO(percorso_modello_best)

# --- 4. TEST SULLE IMMAGINI NORMALI ---
print("\n" + "=" * 50)
print(" 1. TEST SUL SET NORMALE (Ciclo 3)")
print("=" * 50)
risultati_normal = modello.val(
    data = "/inserisci/il/path/per/dataset_test_normal_ciclo3.yaml", # Path al file YAML che abbiamo generato prima
    split = "val",
    imgsz = 1024,
    batch = 8,
    device = 0,
    conf = 0.001,      # Fondamentale per la metrica AR_0.5 accademica
    iou = 0.50,        # Fissa l'IoU al 50% in linea con le metriche del paper YOLOX
    max_det = 100,     # Standard COCO per calcolare l'Average Recall
    augment = True,
    name = "test_normal_metrics_ciclo3"
)

# --- 5. TEST SULLE IMMAGINI FISHEYE (MASSIMIZZATO) ---
print("\n" + "=" * 50)
print(" 2. TEST SUL SET FISHEYE (Ciclo 3)")
print("=" * 50)
risultati_fisheye = modello.val(
    data = "/inserisci/il/path/per/dataset_test_fisheye_ciclo3.yaml", # Path al file YAML che abbiamo generato prima
    split = "val",
    imgsz = 1024,
    batch = 8,
    device = 0,
    conf = 0.001,
    iou = 0.50,
    max_det = 100,
    augment = True,
    name = "test_fisheye_metrics_ciclo3"
)

# --- 6. STAMPA DELLE METRICHE UFFICIALI (FORMATO PAPER) ---
print("\n" + "=" * 50)
print(" RISULTATI (CICLO 3 - DATA CENTRIC)")
print("=" * 50)

# In YOLO, l'indice 0 è la prima classe ("Volto"), l'indice 1 è la seconda ("Targa")
idx_volto = 0
idx_targa = 1

# Estrazione programmatica per lo scenario Normale
ap50_face_norm = risultati_normal.box.ap50[idx_volto] * 100
ar50_face_norm = risultati_normal.box.r[idx_volto] * 100
ap50_plate_norm = risultati_normal.box.ap50[idx_targa] * 100
ar50_plate_norm = risultati_normal.box.r[idx_targa] * 100

# Estrazione programmatica per lo scenario Fisheye
ap50_face_fish = risultati_fisheye.box.ap50[idx_volto] * 100
ar50_face_fish = risultati_fisheye.box.r[idx_volto] * 100
ap50_plate_fish = risultati_fisheye.box.ap50[idx_targa] * 100
ar50_plate_fish = risultati_fisheye.box.r[idx_targa] * 100

print("\n--- SCENARIO NORMALE (Test Set Massimo) ---")
print(f"AP_0.5 (face): {ap50_face_norm:.2f}%")
print(f"AR_0.5 (face): {ar50_face_norm:.2f}%")
print(f"AP_0.5 (plate): {ap50_plate_norm:.2f}%")
print(f"AR_0.5 (plate): {ar50_plate_norm:.2f}%")

print("\n--- SCENARIO FISHEYE (Test Set Massimo) ---")
print(f"AP_0.5 (face): {ap50_face_fish:.2f}%")
print(f"AR_0.5 (face): {ar50_face_fish:.2f}%")
print(f"AP_0.5 (plate): {ap50_plate_fish:.2f}%")
print(f"AR_0.5 (plate): {ar50_plate_fish:.2f}%")

# --- 7. STAMPA METRICHE TRADIZIONALI GLOBALI (MATRICE COMPLETA) ---
print("\n" + "=" * 50)
print(" METRICHE TRADIZIONALI GLOBALI (COMPRENSIVE)")
print("=" * 50)

print("\n[NORMALE] Metriche Medie (Tutte le classi combinate):")
print(f"- Precision (P): {risultati_normal.box.mp * 100:.2f}%")
print(f"- Recall (R): {risultati_normal.box.mr * 100:.2f}%")
print(f"- mAP@0.5: {risultati_normal.box.map50 * 100:.2f}%")
print(f"- mAP@0.5:0.95: {risultati_normal.box.map * 100:.2f}%")

print("\n[FISHEYE] Metriche Medie (Tutte le classi combinate):")
print(f"- Precision (P): {risultati_fisheye.box.mp * 100:.2f}%")
print(f"- Recall (R): {risultati_fisheye.box.mr * 100:.2f}%")
print(f"- mAP@0.5: {risultati_fisheye.box.map50 * 100:.2f}%")
print(f"- mAP@0.5:0.95: {risultati_fisheye.box.map * 100:.2f}%")